# M10 lab: forecasting load, wind and solar, judged honestly

*Energy Systems, Step by Step* · Modeler tier · lab for lesson M10 · about 60 minutes, plus 90 minutes of practice at the end.

You will rebuild the lesson's forecasts in pandas and check them against the page with `assert` cells, then go further than a browser can:

1. The data and the clock: SPP's hourly demand in 2025, a forecast made at local midnight for each hour of the next day.
2. Baselines first: persistence, same hour yesterday, same hour last week.
3. A linear model on features known at midnight; 16 feature mixes chosen on October; two leaks.
4. A band from October's misses, scored on November with coverage and pinball loss.
5. Gradient boosting and a small neural network on the same hours; quantile forecasts; a rolling-origin check.

**How to run it.** Put this notebook, `m10-ml-forecasting.json` (*Lesson settings*) and `town_year.json` (*Hourly data*) in one folder. In Google Colab: *File → Upload notebook*, then upload the two data files with the folder icon on the left, then *Runtime → Run all*. Locally: Python 3.9 or later with `pandas`, `numpy` and `scikit-learn`. Everything runs in about a minute.

**The data.** Real hourly demand, wind and solar of the Southwest Power Pool as reported to EIA-930 for 2025, as cleaned for the course (rules in the file's `meta`). Each series is stored in thousandths of its 2025 maximum; demand is rebuilt in MW from SPP's 2025 peak hour, 54,411 MW.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

for f in ("m10-ml-forecasting.json", "town_year.json"):
    assert Path(f).exists(), f + " not found: put it next to this notebook (in Colab, upload it with the folder icon)."
D = json.load(open("m10-ml-forecasting.json"))
Y = json.load(open("town_year.json"))
REF = D["reference"]
print(Y["meta"]["source"])

## 1. The clock and the split

Index $i$ of `town_year.json` is the hour that **ends** at `start_utc` + $i$ hours (UTC). The page reads it in Central time with daylight saving, using one switch rule: an hour ending after the spring switch and no later than the autumn switch is in daylight time. An hour ending at midnight belongs to the day before (hour ending 24). The forecast for a day is made at the last hour of the day before; the horizon $h$ is the number of hours since then.

In [ ]:
CL = D["clock"]
t_end = pd.Timestamp(Y["start_utc"], tz="UTC") + pd.to_timedelta(np.arange(len(Y["demand"])), unit="h")
dst = (t_end > pd.Timestamp(CL["dstStartUTC"])) & (t_end <= pd.Timestamp(CL["dstEndUTC"]))
local = (t_end + pd.to_timedelta(np.where(dst, CL["offsetDST"], CL["offsetStd"]), unit="h")).tz_localize(None)
he = local.hour.to_numpy().copy()
day = local.normalize()
day = day.where(he != 0, day - pd.Timedelta(days=1))
he[he == 0] = 24
df = pd.DataFrame({"date": day.strftime("%Y-%m-%d"), "he": he, "dow": (day.day_of_week.to_numpy() + 1) % 7,   # 0 = Sunday, as the page
                   "demand": np.array(Y["demand"]) / 1000 * D["peakMW"], "wind": Y["wind"], "solar": Y["solar"]})
df["issue"] = df["date"].map(df.reset_index().groupby("date")["index"].first()) - 1
df["h"] = df.index - df["issue"]
print(df.groupby("date").size().value_counts().to_dict(), "(hours per day: count of days)")
S = D["split"]
part = lambda r: df[(df["date"] >= r[0]) & (df["date"] <= r[1])]
train, val, test = part(S["train"]), part(S["val"]), part(S["test"])
assert (len(train), len(val), len(test)) == (REF["hours"]["train"], REF["hours"]["val"], REF["hours"]["test"])
print(f"train {len(train)} hours, choose on October {len(val)}, test on November {len(test)}")

## 2. Baselines first

- **Persistence**: the latest actual, held flat: $\hat y_t = y_{\text{issue}}$.
- **Same hour yesterday**: $\hat y_t = y_{t - 24\lceil h/24\rceil}$ (lag 48 for the 25th hour of 2 November, so no input looks ahead).
- **Same hour last week**: $\hat y_t = y_{t-168}$.

In [ ]:
y = df["demand"].to_numpy()
lagday = 24 * np.ceil(df["h"].to_numpy() / 24).astype(int)
RULES = {"persist": lambda s: y[s["issue"].to_numpy()],
         "day": lambda s: y[s.index.to_numpy() - lagday[s.index.to_numpy()]],
         "week": lambda s: y[s.index.to_numpy() - 168]}
mae = lambda s, f: np.mean(np.abs(y[s.index] - f))
for k, f in RULES.items():
    v, t = mae(val, f(val)), mae(test, f(test))
    assert abs(v - REF["baselines"][k]["val"]) < 1e-6 and abs(t - REF["baselines"][k]["test"]) < 1e-6
    print(f"{k:8s} October MAE {v:6.0f} MW   November {t:6.0f} MW")

## 3. A linear model on features known at midnight

The design has an intercept, same hour yesterday, and the optional groups the page offers: same hour last week, the latest actual, hour of day (23 indicators) and Saturday and Sunday indicators (inputs in GW). Least squares on the training months; the 16 mixes are ranked by October's MAE. Then the two leaks: the hour before, and tomorrow's average demand.

In [ ]:
idx = df.index.to_numpy()
cols = {"const": np.ones(len(df)), "day": y[np.maximum(idx - lagday, 0)] / 1000, "week": y[np.maximum(idx - 168, 0)] / 1000,
        "last": y[np.maximum(df["issue"].to_numpy(), 0)] / 1000, "lag1": y[np.maximum(idx - 1, 0)] / 1000,
        "oracle": df.groupby("date")["demand"].transform("mean").to_numpy() / 1000}
hour = {f"he{h}": (df["he"].to_numpy() == h).astype(float) for h in range(2, 25)}
wk = {"sat": (df["dow"].to_numpy() == 6).astype(float), "sun": (df["dow"].to_numpy() == 0).astype(float)}
GROUP = {"week": ["week"], "last": ["last"], "hour": list(hour), "wkend": list(wk), "lag1": ["lag1"], "oracle": ["oracle"]}
ALL = {**cols, **hour, **wk}

def design(rows, use):
    names = ["const", "day"] + [c for g in use for c in GROUP[g]]
    return np.column_stack([ALL[c][rows.index] for c in names])

def fit(use):
    w, *_ = np.linalg.lstsq(design(train, use), y[train.index], rcond=None)
    return lambda rows: design(rows, use) @ w

OPT = ["week", "last", "hour", "wkend"]
mixes = []
for m in range(16):
    use = [g for j, g in enumerate(OPT) if m >> j & 1]
    f = fit(use)
    mixes.append(("+".join(use) or "day", mae(val, f(val)), mae(test, f(test)), use))
mixes.sort(key=lambda r: r[1])
for (k, v, t, _), (rk, rv, rt) in zip(mixes, REF["mixes"]):
    assert k == rk and abs(v - rv) < 1e-4 and abs(t - rt) < 1e-4, (k, rk)
best = mixes[0]
print(f"best on October: {best[0]}, {best[1]:.0f} MW (same hour yesterday {REF['baselines']['day']['val']:.0f}); November {best[2]:.0f} MW")
for leak in ("lag1", "oracle"):
    v = mae(val, fit(best[3] + [leak])(val))
    assert abs(v - REF["leaks"][leak]) < 1e-4
    print(f"+ {leak:6s}: {v:.0f} MW on October, from a value nobody has at midnight")

## 4. A band from October's misses

Quantiles (linear interpolation, as `numpy.quantile` does by default) of the chosen model's October misses give the band: forecast + $q_\tau$. Coverage counts November's hours inside the 80% band ($q_{0.1}$ to $q_{0.9}$); the pinball loss averages over the 19 levels 5% to 95%.

In [ ]:
fb = fit(best[3])
TAU = np.arange(1, 20) / 20
Q = np.quantile(y[val.index] - fb(val), TAU)
assert np.allclose(Q, REF["quantiles"], atol=1e-4)
m, yt = fb(test), y[test.index]
cov80 = np.mean((yt >= m + Q[1]) & (yt <= m + Q[17]))
q = m[:, None] + Q[None, :]
pin = np.mean(np.where(yt[:, None] >= q, TAU * (yt[:, None] - q), (1 - TAU) * (q - yt[:, None])))
assert abs(cov80 - REF["band"]["cov80"]) < 1e-9 and abs(pin - REF["band"]["pinball"]) < 1e-4
print(f"80% band {Q[1]:.0f} to +{Q[17]:.0f} MW; holds {100 * cov80:.1f}% of November; pinball loss {pin:.1f} MW")

## 5. Gradient boosting and a small neural network

Both get the same honest inputs as numbers (same hour yesterday, same hour last week, the latest actual, hour of day, day of week), the same training months, and are chosen on October against the linear mix. Only the winner on October is then scored on November. The neural network needs scaled inputs and a scaled target; both models are seeded.

In [ ]:
from sklearn.compose import TransformedTargetRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

feat = lambda rows: np.column_stack([cols["day"][rows.index], cols["week"][rows.index], cols["last"][rows.index], rows["he"], rows["dow"]])
models = {
    "gradient boosting": HistGradientBoostingRegressor(max_iter=300, learning_rate=0.05, random_state=0),
    "neural network (16 units)": TransformedTargetRegressor(make_pipeline(StandardScaler(), MLPRegressor(hidden_layer_sizes=(16,), max_iter=2000, early_stopping=True, random_state=0)), transformer=StandardScaler()),
}
scores = {"linear (page's mix)": (best[1], fb)}
for name, mdl in models.items():
    mdl.fit(feat(train), y[train.index])
    scores[name] = (mae(val, mdl.predict(feat(val))), lambda rows, mdl=mdl: mdl.predict(feat(rows)))
base_v, base_t = REF["baselines"]["day"]["val"], REF["baselines"]["day"]["test"]
for name, (v, _) in scores.items():
    print(f"{name:26s} October MAE {v:6.0f} MW  skill {100 * (1 - v / base_v):5.1f}%")
winner = min(scores, key=lambda k: scores[k][0])
t = mae(test, scores[winner][1](test))
print(f"chosen on October: {winner}; November MAE {t:.0f} MW, skill {100 * (1 - t / base_t):.1f}% against same hour yesterday")
assert all(np.isfinite(v) and v < 2 * base_v for v, _ in scores.values())

**Read it.** Whether a flexible model beats the linear mix here depends on the month, and none of them sees temperature, the input real load forecasters rely on. Quantile versions of gradient boosting give a band directly:

In [ ]:
lo = HistGradientBoostingRegressor(loss="quantile", quantile=0.1, max_iter=300, learning_rate=0.05, random_state=0).fit(feat(train), y[train.index])
hi = HistGradientBoostingRegressor(loss="quantile", quantile=0.9, max_iter=300, learning_rate=0.05, random_state=0).fit(feat(train), y[train.index])
l, h_ = lo.predict(feat(test)), hi.predict(feat(test))
pl = lambda tau, qv: np.mean(np.where(yt >= qv, tau * (yt - qv), (1 - tau) * (qv - yt)))
print(f"quantile boosting, 80% band: holds {100 * np.mean((yt >= l) & (yt <= h_)):.1f}% of November, mean width {np.mean(h_ - l):.0f} MW; "
      f"pinball at 10% and 90%: {pl(0.1, l):.0f} and {pl(0.9, h_):.0f} MW")
print(f"the page's band: holds {100 * cov80:.1f}%, width {Q[17] - Q[1]:.0f} MW; pinball at 10% and 90%: {pl(0.1, m + Q[1]):.0f} and {pl(0.9, m + Q[17]):.0f} MW")

## 6. One test month is a small sample: a rolling origin

Refit the page's mix on everything before each month from May to November, and test on that month alone. The skill against same hour yesterday moves from month to month; report its spread, not one number.

In [ ]:
rows = []
for mth in range(5, 12):
    start = f"2025-{mth:02d}-01"
    end = (pd.Timestamp(start) + pd.offsets.MonthEnd(0)).strftime("%Y-%m-%d")
    tr = df[(df["date"] >= "2025-01-08") & (df["date"] < start)]
    te = df[(df["date"] >= start) & (df["date"] <= end)]
    w, *_ = np.linalg.lstsq(design(tr, best[3]), y[tr.index], rcond=None)
    mm = mae(te, design(te, best[3]) @ w); bb = mae(te, RULES["day"](te))
    rows.append((start[:7], round(mm), round(bb), round(100 * (1 - mm / bb), 1)))
print(pd.DataFrame(rows, columns=["month", "MAE model", "MAE yesterday", "skill %"]).to_string(index=False))

## 7. Practice (about 90 minutes)

Write your own cells below each task. Predict first, then run; report every score with its months, its baseline and its horizon.

1. **Wind, one to six hours ahead** (about 20 minutes). Forecast `wind` at horizons 1 to 6 with a linear model on its last three hours (lags at least the horizon). At which horizon does it stop beating persistence on October?
2. **A holiday feature** (about 15 minutes). Add an indicator for U.S. federal holidays (for example Labor Day, 1 September, and Thanksgiving, 27 November). Choose on October as before: does the feature survive the choice, and what does it do to November's worst days?
3. **Solar without MAPE** (about 15 minutes). Score same hour yesterday and a linear model with the last week's same hours for `solar`, as MAE in % of the maximum, over all hours and over daylight hours only. Why do the two answers differ, and which would you report?
4. **A fair contest** (about 20 minutes). Run the rolling origin of section 6 for the linear mix, gradient boosting and the neural network. Which wins most months, and by how much more than the month-to-month spread?
5. **Write it up** (about 20 minutes). In six sentences, report a day-ahead demand forecast as a modeler would: the data and its cleaning, the clock and the horizon, the split, the baseline and the skill on the test month, the band's coverage and pinball loss, and the worst days.